# 06 — Streamlit Prediction Pipeline Demo (3 classes)

Demonstrates the **exact prediction pipeline the Streamlit app uses**, end-to-end, on a
real Apple-Watch `WristMotion.csv` sample — without launching the web UI.

It loads the committed model + feature names and calls `predict_from_sensor_logger()`
from `ml4b.data.apple_watch_loader`, then shows the per-window predictions (the three
exercises plus gated **rest** and **uncertain**), confidence, and the detected sampling
rate — the same numbers the app renders on its Predict page.

Every heavy step is delegated to `src/ml4b/`; this notebook only orchestrates and
visualises.

In [ ]:
import joblib
import matplotlib.pyplot as plt

from ml4b.data.apple_watch_loader import predict_from_sensor_logger
from ml4b.data.canonical import CONFIDENCE_THRESHOLD, OVERLAP, TARGET_HZ, WINDOW_SIZE
from ml4b.utils.config import DATA_PROCESSED, DATA_RAW, MODELS_DIR

model = joblib.load(MODELS_DIR / "best_model.joblib")
feature_names = (DATA_PROCESSED / "feature_names.txt").read_text().strip().split("\n")
print("model classes      :", list(model.classes_))
print("features           :", len(feature_names))
print(
    f"window {WINDOW_SIZE} @ {TARGET_HZ} Hz, overlap {OVERLAP}, conf threshold {CONFIDENCE_THRESHOLD}"
)

## 1. Run the pipeline on a sample
We use a bundled Apple-Watch sample. Uploading this same `WristMotion.csv` (or its ZIP)
on the app's Predict page produces identical results.

In [ ]:
sample = DATA_RAW / "apple_watch" / "test_samples" / "bicep_curl_sample_2.csv"
print("sample:", sample.name)
results = predict_from_sensor_logger(sample, model, feature_names)
print("detected rate :", results.attrs.get("detected_hz"), "Hz")
print("windows       :", len(results))
results.head()

## 2. Per-window distribution

In [ ]:
dist = results["predicted_class"].value_counts()
print(dist.to_string())
ex = results[~results["predicted_class"].isin({"rest", "uncertain"})]
top = ex["predicted_class"].mode().iloc[0] if not ex.empty else "none"
conf = (
    float(results["confidence"].mean())
    if results["confidence"].notna().any()
    else float("nan")
)
print(f"\ntop exercise (excl. rest/uncertain): {top}")
print(f"avg confidence (classified)        : {conf:.3f}")

## 3. Timeline
Each 2-second window over time, coloured by predicted label (rest/uncertain in grey) —
the same view the app shows.

In [ ]:
colors = {
    "bicep_curl": "#4C78A8",
    "tricep_extension": "#E45756",
    "row": "#54A24B",
    "rest": "#c9ccd1",
    "uncertain": "#8a8f98",
}
fig, ax = plt.subplots(figsize=(10, 1.8))
step = WINDOW_SIZE * (1 - OVERLAP) / TARGET_HZ
for _, r in results.iterrows():
    ax.barh(
        0,
        step,
        left=r["time_start_seconds"],
        height=1,
        color=colors.get(r["predicted_class"], "#999999"),
    )
ax.set_yticks([])
ax.set_xlabel("time (s)")
ax.set_title("Predicted exercise timeline")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(
    handles,
    colors.keys(),
    ncol=5,
    fontsize=8,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.5),
)
plt.tight_layout()
plt.show()

## 4. Notes
- `rest` is produced by the energy gate (ADR-017), `uncertain` by the confidence
  threshold (ADR-020) — neither is a trained class.
- This is the identical code path as the app, so the demo and the app agree exactly.

## 5. Summary
The deployed pipeline turns a raw Apple-Watch `WristMotion.csv` into per-window exercise
predictions with confidence, using only committed artifacts (model + feature names) —
no dataset required. Launch the full UI with `make run` (or
`uv run streamlit run app/streamlit_app.py`).